# How well are European day-ahead power prices coupled, and where do they break apart?

Data: ENTSO-E Transparency Platform (day-ahead prices, day-ahead load and wind and solar forecasts,
physical cross-border flows, day-ahead NTC) and FRED/IMF monthly TTF gas prices.

To reproduce from scratch:
```bash
export ENTSOE_API_KEY=...          # or put it in ~/.entsoe_key
python -m coupling.fetch           # about 1 hour on the first run; results cached in data/raw/
python -m coupling.clean           # builds data/processed/
jupyter nbconvert --execute --to notebook --inplace notebooks/analysis.ipynb
```

Conventions:
* All timestamps are UTC internally, so DST never creates duplicate or missing hours. Hour-of-day and
  delivery day use market time (Europe/Brussels).
* A border-hour counts as **converged** when |p_a − p_b| < 1 €/MWh. This is ACER's "full price
  convergence" threshold. Exact convergence (< 0.01 €/MWh) is reported as a sensitivity check.
* After 1 Oct 2025 the hourly price is the mean of the four 15-minute prices, and only complete hours
  are kept.

In [ ]:
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from coupling import config as cfg, clean, metrics, models, events, plots
from coupling.fetch import fetch_gas

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.float_format", "{:.3f}".format)
plt.rcParams["figure.dpi"] = 100

data = clean.load_processed()
prices, fund, flows, ntc = data["prices"], data["fundamentals"], data["flows"], data["ntc"]
gas = fetch_gas()
RESULTS = cfg.ROOT / "results"
RESULTS.mkdir(exist_ok=True)
summary = {}
print(prices.index.min(), "→", prices.index.max(), "|", prices.shape)

## 1. Data quality

Checks: missing or incomplete hours per zone, conflicting duplicate stamps, DST-day completeness,
currency and unit as stated in the ENTSO-E XML, negative prices, and interconnector start dates.

In [ ]:
q = data["quality"]
display(q[["n_hours", "n_missing", "n_incomplete_hours", "n_exact_duplicates", "n_conflicting_stamps",
           "days_incomplete", "dst_days_23h", "dst_days_25h", "n_negative", "min", "max"]])
q.to_csv(RESULTS / "price_quality.csv")
summary["max_missing_share"] = float((q["n_missing"] / q["n_hours"]).max())

In [ ]:
cur = pd.read_csv(cfg.RAW_DIR / "currency_audit.csv")
display(cur.groupby(["currency", "unit"], dropna=False).size().rename("zone-years").to_frame())
assert set(cur["currency"].dropna()) == {"EUR"}, "non-EUR price series found — convert before comparing"
assert set(cur["unit"].dropna()) == {"MWH"}

In [ ]:
neg = (prices < 0).groupby(prices.index.tz_convert(cfg.MARKET_TZ).year).sum()
display(neg.rename(columns=plots.ZONE_LABEL))
summary["negative_hours_by_year"] = neg.sum(axis=1).to_dict()

In [ ]:
starts = metrics.border_start(flows)
display(starts.to_frame())

In [ ]:
# Crisis window: the first contiguous run since 2021 of months with TTF > 3x its 2015-2020 mean.
# The assert checks that config.py matches the data.
base = gas["2015":"2020"].mean()
hot = (gas > cfg.CRISIS_TTF_MULTIPLE * base)["2021":]
run_start = hot.idxmax()
run_end = hot[run_start:][~hot[run_start:]].index[0]
print("TTF baseline", round(base, 2), "USD/MMBtu; crisis run:", run_start.date(), "→", run_end.date(), "(exclusive)")
assert run_start == cfg.CRISIS_START and run_end == cfg.CRISIS_END
fig, ax = plt.subplots(figsize=(10, 3))
gas["2018":].plot(ax=ax); ax.axhline(cfg.CRISIS_TTF_MULTIPLE * base, ls=":", c="grey")
ax.axvspan(cfg.CRISIS_START.tz_localize(None), cfg.CRISIS_END.tz_localize(None), color="grey", alpha=0.1)
ax.set_ylabel("TTF, USD/MMBtu"); ax.set_title("Monthly TTF and the crisis window");

## 2. Q1: How often do neighbouring zones clear at the same price?

In [ ]:
panel = metrics.border_panel(prices, starts=starts)
tbl = (panel.groupby("border")
       .agg(hours=("conv", "size"), conv_acer=("conv", "mean"), conv_exact=("conv_exact", "mean"),
            mean_abs_spread=("abs_spread", "mean"), median_abs_spread=("abs_spread", "median"))
       .sort_values("conv_acer", ascending=False))
display(tbl)
tbl.to_csv(RESULTS / "border_convergence.csv")
summary["border_convergence"] = tbl[["conv_acer", "conv_exact", "mean_abs_spread"]].to_dict("index")
summary["acer_vs_exact_max_gap_pp"] = float((tbl["conv_acer"] - tbl["conv_exact"]).max() * 100)

In [ ]:
feats = metrics.market_time_features(prices.index)
period = pd.Series(np.select([feats["crisis"], prices.index < cfg.CRISIS_START],
                             ["2 crisis", "1 pre-crisis"], "3 post-crisis"), index=prices.index)
plots.heatmap(metrics.pairwise_matrix(prices), "All zone pairs: % hours converged, full sample", save="heatmap_full")
for p in sorted(period.unique()):
    plots.heatmap(metrics.pairwise_matrix(prices, mask=period == p), f"% hours converged — {p[2:]}",
                  save=f"heatmap_{p[2:].replace('-', '_')}")

In [ ]:
qshare = metrics.share_by(panel, ["border", "quarter"])
core_borders = [b.name for b in cfg.BORDERS if b.ccr == "Core"]
plots.quarterly_lines(qshare[qshare["border"].isin(core_borders)], "Core borders: convergence by quarter",
                      save="quarterly_core")
plots.quarterly_lines(qshare[~qshare["border"].isin(core_borders)], "Peripheral and Nordic borders: convergence by quarter",
                      save="quarterly_periphery");

In [ ]:
core4 = metrics.region_convergence(prices, cfg.CORE_ZONES)
core4_year = core4.groupby(feats["year"]).mean()
display(core4_year.rename("NL-BE-DE_LU-FR all within 1 €/MWh").to_frame())
summary["core4_full_convergence_by_year"] = core4_year.to_dict()

In [ ]:
plots.spread_histograms(panel, [b.name for b in cfg.BORDERS], save="spread_histograms");

## 3. Q2: Does non-convergence cluster in specific hours and regimes?

Regimes, defined in `config.py`:
* **crisis**: Sep 2021 to Feb 2023, the months when TTF was more than 3x its 2015-2020 mean.
* **winter peak**: Dec-Feb, working days, 17:00-20:59.
* **high-RES**: the top decile, within each calendar year, of combined day-ahead wind+solar forecast
  divided by load forecast across the two zones of a border.

In [ ]:
hshare = metrics.share_by(panel[~panel.index.isin(prices.index[feats["crisis"]])], ["border", "hour"])
plots.hour_profile(hshare[hshare["border"].isin(core_borders + ["FR-ES", "DE_LU-DK_1"])], save="hour_profile");

In [ ]:
rows = []
for b in cfg.BORDERS:
    d = panel[panel["border"] == b.name]
    f = feats.loc[d.index]
    hr = metrics.high_res_flag(fund, b.a, b.b).reindex(d.index)
    nc = 1 - d["conv"]
    rows.append({"border": b.name,
                 "all hours": nc.mean(),
                 "crisis": nc[f["crisis"].to_numpy()].mean(),
                 "non-crisis": nc[~f["crisis"].to_numpy()].mean(),
                 "winter peak": nc[f["winter_peak"].to_numpy()].mean(),
                 "high-RES (top decile)": nc[(hr == 1).to_numpy()].mean(),
                 "other RES (bottom 90%)": nc[(hr == 0).to_numpy()].mean()})
regimes = pd.DataFrame(rows).set_index("border")
display(regimes)
regimes.to_csv(RESULTS / "nonconvergence_by_regime.csv")
plots.regime_bars(regimes.drop(columns=["all hours"]), save="regimes")
summary["nonconv_by_regime_median_across_borders"] = regimes.median().to_dict()

### Explanatory regressions

Unit: border-hour. LPM for 1[|Δp| ≥ 1 €/MWh] and OLS for |Δp|. Fixed effects: border, hour, weekday,
month. Standard errors are clustered by delivery day. Regressors are z-scored.

* **A, fundamentals**: forecast RES share on each side, forecast load relative to its annual mean on
  each side, log TTF.
* **B, + congestion**: A plus flow utilisation, i.e. |net flow| relative to the border's quarterly
  99th percentile. Congestion is how prices separate, so B says how much the congestion proxy
  absorbs. It is not a causal estimate.

Which side is `a` follows the border name, e.g. `FR-ES`: a = FR.

In [ ]:
rf = models.regression_frame(prices, fund, flows, ntc, gas, starts=starts)
fits = {
    "A: P(nonconv)": models.fit(rf, "nonconv", models.SPEC_A),
    "B: P(nonconv)": models.fit(rf, "nonconv", models.SPEC_B),
    "A: |Δp|": models.fit(rf, "abs_spread", models.SPEC_A),
    "B: |Δp|": models.fit(rf, "abs_spread", models.SPEC_B),
}
terms = ["res_a", "res_b", "load_rel_a", "load_rel_b", "log_ttf", "flow_util"]
ct = models.coef_table(fits, terms)
display(ct)
ct.to_csv(RESULTS / "regressions_pooled.csv")
summary["pooled_regressions"] = {k: {"params": v.params[[t for t in terms if t in v.params]].to_dict(),
                                     "r2": v.rsquared} for k, v in fits.items()}

In [ ]:
pb = models.per_border(rf, "nonconv", models.SPEC_B)
pb.to_csv(RESULTS / "regressions_per_border.csv")
display(pb[["N", "mean_outcome", "R2", "res_a", "res_b", "load_rel_a", "load_rel_b", "flow_util"]])
plots.coef_plot(pb, ["res_a", "res_b", "load_rel_a", "load_rel_b", "flow_util"], save="coef_per_border");

In [ ]:
# Robustness for NTC borders: utilisation against the published day-ahead NTC, not the flow-quantile proxy.
ntc_borders = [b.name for b in cfg.BORDERS if b.regime == "NTC"]
rf_ntc = rf[rf["border"].isin(ntc_borders) & rf["ntc_util"].notna()]
if len(rf_ntc) > 10_000:
    r_ntc = models.fit(rf_ntc, "nonconv", models.SPEC_A + " + ntc_util")
    display(models.coef_table({"NTC borders, NTC utilisation": r_ntc}, terms[:-1] + ["ntc_util"]))
    summary["ntc_util_coef"] = float(r_ntc.params["ntc_util"])
else:
    print("too few hours with published day-ahead NTC:", len(rf_ntc))

## 4. Q3: Did the 15-minute MTU change anything?

Descriptive only. The switch hit every zone on the same day, so there is no control group. The
comparison is 12 months before vs 12 months after (Oct-Sep in both cases, which matches the seasons).
The 95% CI comes from a bootstrap over delivery days. Other things also changed between the two
windows: fuel prices, new RES capacity, and Nordic flow-based, which is inside the pre-window for the
Nordic borders. Read the differences as changes over time, not as effects of the switch.

The 15-minute data also shows something hourly data can't: hours whose **hourly mean** converges
even though some quarter-hours do not.

In [ ]:
ba = events.mtu15_before_after(panel)
display(ba)
ba.to_csv(RESULTS / "mtu15_before_after.csv")
plots.mtu15_plot(ba, save="mtu15_before_after")
summary["mtu15_before_after"] = ba[["pre", "post", "diff", "ci_lo", "ci_hi"]].to_dict("index")

In [ ]:
if "prices_15min" in data:
    rows = []
    for b in cfg.BORDERS:
        qc = metrics.quarter_hour_convergence(data["prices_15min"], b.a, b.b)
        if len(qc) == 0:
            continue
        rows.append({"border": b.name, "hours": len(qc),
                     "conv (hourly mean)": qc["conv_hourly_mean"].mean(),
                     "conv (all 4 quarters)": qc["conv_all_quarters"].mean(),
                     "hidden divergence": ((qc["conv_hourly_mean"] == 1) & (qc["conv_all_quarters"] == 0)).mean(),
                     "median intra-hour spread range": qc["intrahour_spread_range"].median()})
    qh = pd.DataFrame(rows).set_index("border")
    display(qh)
    qh.to_csv(RESULTS / "mtu15_intrahour.csv")
    summary["mtu15_intrahour"] = qh.to_dict("index")

## 5. Exploratory: flow-based go-lives as natural experiments

The unit is the border-day, and the outcome is the share of converged hours. The model is TWFE with
border and date fixed effects. The p-value is exact: it is the share of all possible assignments of
the "treated" label to borders that produce an |effect| at least as large. With about 10 borders,
clustered standard errors would not be trustworthy.

* **Core FB, delivery 9 Jun 2022.** The treated CWE borders were already flow-based (CWE FB, since
  2015), so this treatment is an extension of the flow-based domain plus the Core minRAM rules, not a
  switch from NTC to FB. Go-live came in the middle of the gas crisis and the French nuclear outage,
  both of which hit treated and control borders unevenly. DE_LU-PL is excluded because it entered
  implicit coupling in Jun 2021.
* **Nordic FB, delivery 30 Oct 2024.** Treated: DK1-SE3 and DK1-NO2, which switched from NTC to FB.
  Controls: Core and SWE borders. Hansa borders are excluded because they are partially treated
  through advanced hybrid coupling. The window ends before the 15-minute switch.

In [ ]:
dd = events.daily(panel)
did_res = {}
for name, design in [("Core FB (2022-06-09)", cfg.CORE_DID), ("Nordic FB (2024-10-30)", cfg.NORDIC_DID)]:
    r = events.did(dd, design)
    did_res[name] = r
    print(f"{name}: beta = {r['beta']*100:+.1f} pp convergence, permutation p = {r['p_perm']:.3f} "
          f"({r['n_perm']} assignments, {r['n_borders']} borders, {r['n_obs']} border-days)")
    display(r["means"].rename(index={0.0: "control", 1.0: "treated"}, columns={0.0: "pre", 1.0: "post"}))
summary["did"] = {k: {"beta": v["beta"], "p_perm": v["p_perm"], "n_perm": v["n_perm"]} for k, v in did_res.items()}

In [ ]:
for name, design, fn in [("Core FB", cfg.CORE_DID, "event_core_fb"), ("Nordic FB", cfg.NORDIC_DID, "event_nordic_fb")]:
    es = events.event_study(dd, design)
    es.to_csv(RESULTS / f"{fn}.csv")
    plots.event_study_plot(es, f"{name}: treated − control convergence by month", save=fn)

## 6. Numbers behind the README

In [ ]:
def _clean(o):
    if isinstance(o, dict):
        return {str(k): _clean(v) for k, v in o.items()}
    if isinstance(o, (np.floating, float)):
        return None if np.isnan(o) else round(float(o), 4)
    if isinstance(o, (np.integer,)):
        return int(o)
    return o
summary["sample"] = {"start": str(prices.index.min()), "end": str(prices.index.max()),
                     "zones": len(prices.columns), "borders": int(panel["border"].nunique()),
                     "years": round((prices.index.max() - prices.index.min()).days / 365.25, 1)}
(RESULTS / "summary.json").write_text(json.dumps(_clean(summary), indent=2))
print(json.dumps(_clean(summary["sample"]), indent=2))